# CASPER Training Pipeline

**CASPER: Continuous Action Space Preference Elicitation via Reinforcement**

Complete training pipeline:
1. Two-tower recommender training
2. RL actor supervised pretraining (Reddit data)
3. End-to-end RL training with per-turn NDCG rewards
4. Baseline evaluation

## 1. Environment Setup

In [ ]:
import sys
import os
import random
from pathlib import Path
import torch
import numpy as np
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
load_dotenv(PROJECT_ROOT / '.env')

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError("OPENAI_API_KEY not found in .env file")

# Set all random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Device: {device}")
print(f"Random seeds set: 42")

## 2. Load Configuration

In [ ]:
import yaml

with open(PROJECT_ROOT / 'config' / 'config.yaml', 'r') as f:
    config = yaml.safe_load(f)

PRETRAIN_EPOCHS = config['training_data']['supervised']['epochs']
RL_EPISODES = config['training_data']['reinforcement']['num_episodes']
MAX_TURNS = config['environment']['max_turns']
DATA_DIR = PROJECT_ROOT / 'data' / 'movielens'

print(f"Configuration:")
print(f"  Pretraining epochs: {PRETRAIN_EPOCHS}")
print(f"  RL episodes: {RL_EPISODES}")
print(f"  Max turns: {MAX_TURNS}")
print(f"  Model: {config['models']['question_generator']['model_name']}")

## 3. Initialize Components

In [ ]:
from casper.data.movielens_loader import MovieLensLoader
from casper.models.embedding_space import SentenceBERTEmbeddingSpace
from casper.models.rl_actor_critic import EmbeddingActorCritic
from casper.models.two_tower_recommender import TwoTowerRecommender, MovieCatalog, RecommenderTrainer
from casper.models.preference_extractor import PreferenceExtractor
from casper.agents.user_simulator import UserSimulator
from casper.agents.casper_agent import CASPERAgent
from casper.baselines.random_agent import RandomAgent
from casper.baselines.pure_llm_agent import PureLLMAgent
from casper.evaluation.conversation_evaluator import ConversationalEvaluator
from casper.evaluation import metrics

loader = MovieLensLoader(data_path=str(DATA_DIR), min_ratings=20)
train_users, test_users = loader.load_data(test_split=0.3)

shared_embedding_space = SentenceBERTEmbeddingSpace(str(DATA_DIR))

casper_agent = CASPERAgent(
    movielens_data_path=str(DATA_DIR),
    load_recommender=True,
    embedding_space=shared_embedding_space
)

user_sim = UserSimulator(
    movielens_data_path=str(DATA_DIR),
    min_ratings=config['models']['user_simulator']['min_ratings'],
    max_profiles=config['models']['user_simulator']['max_profiles']
)

random_agent = RandomAgent(
    movielens_data_path=str(DATA_DIR),
    embedding_space=shared_embedding_space
)

llm_agent = PureLLMAgent(
    movielens_data_path=str(DATA_DIR),
    model_name='gpt-4o-mini',
    embedding_space=shared_embedding_space
)

movie_catalog = MovieCatalog(str(DATA_DIR), encoder=casper_agent.encoder)
shared_recommender_model = TwoTowerRecommender(state_dim=384, embedding_dim=128)
shared_recommender = RecommenderTrainer(
    recommender=shared_recommender_model,
    movie_catalog=movie_catalog,
    encoder=casper_agent.encoder
)

shared_pref_extractor = PreferenceExtractor()
evaluator = ConversationalEvaluator(holdout_ratio=0.3, min_rating_threshold=4.0)

print(f"Train users: {len(train_users)}, Test users: {len(test_users)}")

## 4. Train Shared Recommender

Train recommender that will be used by:
1. CASPER agent: For per-turn reward calculation during RL training
2. Baseline agents: For evaluation only (after conversation ends)

This ensures fair comparison - all agents evaluated with same recommender.

In [ ]:
import pandas as pd
import json
from tqdm import tqdm
from casper.data.recommender_data_builder import create_recommender_training_data

rec_checkpoint = CHECKPOINT_DIR / 'shared_recommender.pt'
training_data_cache = CHECKPOINT_DIR / 'recommender_training_data.json'

RECOMMENDER_TRAIN_USERS = config['models']['recommender']['training_users']
RECOMMENDER_EPOCHS = config['models']['recommender']['epochs']
RECOMMENDER_BATCH_SIZE = config['models']['recommender']['batch_size']

# Check if fully trained checkpoint exists
if rec_checkpoint.exists():
    checkpoint = torch.load(rec_checkpoint)
    if checkpoint.get('epoch', 0) >= RECOMMENDER_EPOCHS:
        print(f"Loading fully trained recommender from {rec_checkpoint}")
        shared_recommender.recommender.user_tower.load_state_dict(checkpoint['user_tower'])
        shared_recommender.recommender.item_tower.load_state_dict(checkpoint['item_tower'])
        
        casper_agent.recommender.recommender.user_tower.load_state_dict(checkpoint['user_tower'])
        casper_agent.recommender.recommender.item_tower.load_state_dict(checkpoint['item_tower'])
        
        trained_users = checkpoint.get('num_users', RECOMMENDER_TRAIN_USERS)
        best_ndcg = checkpoint.get('best_val_ndcg', 0.0)
        print(f"Shared recommender loaded (trained on {trained_users} users, best NDCG: {best_ndcg:.4f})")
    else:
        print(f"Found incomplete checkpoint at epoch {checkpoint.get('epoch', 0)}/{RECOMMENDER_EPOCHS}")
        print("Resuming training...")
        rec_checkpoint = None
else:
    rec_checkpoint = None

if rec_checkpoint is None:
    print(f"Training shared recommender on {RECOMMENDER_TRAIN_USERS} users...")
    
    # Check for cached training data
    if training_data_cache.exists():
        print(f"Loading cached training data...")
        with open(training_data_cache, 'r', encoding='utf-8') as f:
            cached_data = json.load(f)
        
        if cached_data['num_users'] == RECOMMENDER_TRAIN_USERS:
            preference_texts = cached_data['preference_texts']
            likes = cached_data['liked_movie_ids']
            dislikes = cached_data.get('disliked_movie_ids', [[] for _ in preference_texts])  # Backward compat
            print(f"Loaded {len(preference_texts)} training examples")
        else:
            print(f"Cache mismatch - regenerating training data")
            cached_data = None
    else:
        cached_data = None
    
    # Create training data if not cached
    if not cached_data:
        print("Creating training data...")
        
        preference_texts, likes, dislikes = create_recommender_training_data(
            ratings_path=DATA_DIR / 'ratings.csv',
            movies_path=DATA_DIR / 'movies.csv',
            genome_scores_path=DATA_DIR / 'genome-scores.csv',
            genome_tags_path=DATA_DIR / 'genome-tags.csv',
            num_users=RECOMMENDER_TRAIN_USERS
        )
        
        # Cache the training data
        with open(training_data_cache, 'w', encoding='utf-8') as f:
            json.dump({
                'num_users': RECOMMENDER_TRAIN_USERS,
                'preference_texts': preference_texts,
                'liked_movie_ids': likes,
                'disliked_movie_ids': dislikes
            }, f, indent=2, ensure_ascii=False)
        print(f"Training data cached to {training_data_cache}")
    
    print(f"\nExample preference: {preference_texts[0]}")
    print(f"Training on {len(preference_texts)} examples for {RECOMMENDER_EPOCHS} epochs\n")
    
    # Train with checkpointing
    shared_recommender.train(
        conversations=preference_texts,
        liked_movies=likes,
        disliked_movies=dislikes,
        epochs=RECOMMENDER_EPOCHS,
        batch_size=RECOMMENDER_BATCH_SIZE,
        checkpoint_path=str(CHECKPOINT_DIR / 'shared_recommender.pt')
    )
    
    # Save metadata
    final_checkpoint = torch.load(CHECKPOINT_DIR / 'shared_recommender.pt')
    final_checkpoint['num_users'] = RECOMMENDER_TRAIN_USERS
    torch.save(final_checkpoint, CHECKPOINT_DIR / 'shared_recommender.pt')
    
    # Load trained weights
    casper_agent.recommender.recommender.user_tower.load_state_dict(
        shared_recommender.recommender.user_tower.state_dict()
    )
    casper_agent.recommender.recommender.item_tower.load_state_dict(
        shared_recommender.recommender.item_tower.state_dict()
    )
    
    print(f"Recommender training complete: {CHECKPOINT_DIR / 'shared_recommender.pt'}")

## 5. Download Reddit Training Data

Download Reddit conversations for supervised actor pretraining.

**Purpose**: Train RL actor to predict good follow-up concepts from user preferences
**Subreddits**: r/MovieSuggestions, r/movies, r/TrueFilm
**Format**: User post → Expert response pairs

In [ ]:
from casper.data.reddit_scraper import RedditScraper
import json
import time

reddit_data_dir = PROJECT_ROOT / 'data' / 'reddit'
reddit_data_dir.mkdir(parents=True, exist_ok=True)

existing_files = list(reddit_data_dir.glob('*_conversations.json'))
total_pairs = 0
if existing_files:
    for json_file in existing_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                pairs = json.load(f)
                total_pairs += len(pairs)
        except:
            pass

if total_pairs > 0:
    print(f"Found existing Reddit data: {total_pairs} conversation pairs in {len(existing_files)} files")
    print(f"Location: {reddit_data_dir}")
else:
    print("Downloading Reddit training data...")
    
    scraper = RedditScraper(output_dir=str(reddit_data_dir), use_semantic_filter=True)
    
    # Download conversation PAIRS (user post + expert response)
    # This is what we need for pretraining the actor
    print(f"\nScraping r/MovieSuggestions (target: 500 conversation pairs)...")
    try:
        pairs = scraper.scrape_conversation_pairs(
            subreddit='MovieSuggestions',
            max_pairs=500
        )
        
        if pairs:
            output_file = reddit_data_dir / 'moviesuggestions_conversations.json'
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(pairs, f, indent=2, ensure_ascii=False)
            
            print(f"  Scraped {len(pairs)} conversation pairs")
            print(f"  Saved to: {output_file}")
        else:
            print(f"  No conversation pairs retrieved")
        
    except Exception as e:
        print(f"  Error scraping: {e}")
        import traceback
        traceback.print_exc()
    
    if len(pairs) > 0:
        print(f"\nReddit data download complete!")
        print(f"  Total conversation pairs: {len(pairs)}")
        print(f"  Saved to: {reddit_data_dir}")
    else:
        print(f"\nWARNING: No Reddit data was downloaded!")
        print(f"  Supervised pretraining will be skipped.")

## 6. RL Actor Supervised Pretraining

Pretrain actor to predict embeddings in correct semantic space using Reddit conversation data.

**Goal:** Learn context → concept embedding mapping BEFORE RL training
**Data:** Reddit conversations (r/MovieSuggestions, r/movies)
**Loss:** MSE between predicted and target concept embeddings

In [ ]:
from casper.data.reddit_loader import RedditDataLoader
from casper.data.batch_reddit_preprocessor import BatchRedditPreprocessor
from tqdm import tqdm
import json

pretrain_checkpoint = CHECKPOINT_DIR / 'actor_pretrained.pt'

if pretrain_checkpoint.exists():
    print(f"Loading pretrained actor from {pretrain_checkpoint}")
    checkpoint = torch.load(pretrain_checkpoint)
    casper_agent.rl_agent.actor.load_state_dict(checkpoint['actor'])
    print(f"Actor loaded from pretrained checkpoint")
else:
    if not reddit_data_dir.exists():
        raise RuntimeError(f"Reddit data directory not found: {reddit_data_dir}")
    
    # Check if we have preprocessed data (with extracted concepts)
    preprocessed_files = list(reddit_data_dir.glob('*_processed.json'))
    
    if not preprocessed_files:
        print(f"No preprocessed Reddit data found. Running batch LLM extraction...")
        print(f"This will extract preferences and concepts using batched async LLM calls.")
        
        # Run batch preprocessing
        preprocessor = BatchRedditPreprocessor(batch_size=50)
        
        # Process all conversation files
        for conv_file in reddit_data_dir.glob('*_conversations.json'):
            output_file = conv_file.parent / f"{conv_file.stem}_processed.json"
            print(f"\nProcessing {conv_file.name}...")
            preprocessor.process_reddit_file(conv_file, output_file)
        
        preprocessed_files = list(reddit_data_dir.glob('*_processed.json'))
    
    if not preprocessed_files:
        raise RuntimeError(f"No preprocessed data found after processing!")
    
    print(f"\nLoading preprocessed Reddit data...")
    
    # Load all preprocessed data
    all_data = []
    for json_file in preprocessed_files:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            all_data.extend(data)
    
    print(f"Loaded {len(all_data)} preprocessed conversation pairs")
    print(f"Creating training examples...")
    
    try:
        training_examples = []
        
        for pair in tqdm(all_data, desc="Processing pairs", unit="pair"):
            user_prefs = pair.get('user_preferences', {})
            response_concepts = pair.get('response_concepts', [])
            
            # Skip if no concepts extracted
            if not response_concepts:
                continue
            
            # Skip if user_preferences is empty
            if not any(user_prefs.get(k, []) for k in ['liked', 'neutral', 'disliked']):
                continue
            
            # Convert preferences to text (same as RL does)
            user_pref_text = casper_agent.preference_extractor.preferences_to_text(user_prefs)
            
            if not user_pref_text:
                continue
            
            # Create training example for each concept
            for concept in response_concepts:  # Use ALL concepts
                concept_emb = casper_agent.embedding_space.get_embedding(concept)
                if concept_emb is not None:
                    training_examples.append({
                        'user_pref_text': user_pref_text,
                        'concept': concept,
                        'target_embedding': concept_emb
                    })
        
        print(f"\nCreated {len(training_examples)} training examples")
        
        if len(training_examples) == 0:
            print("\nWARNING: No valid training examples created!")
            print("  Skipping supervised pretraining - actor will train from scratch during RL.")
        else:
            # Encode all preference texts with SentenceBERT (batch encoding)
            print(f"Encoding preference texts with SentenceBERT...")
            
            pref_texts = [ex['user_pref_text'] for ex in training_examples]
            
            # Batch encode (fast)
            pref_embeddings = casper_agent.encoder.encode(
                pref_texts,
                convert_to_numpy=True,
                show_progress_bar=True,
                batch_size=64
            )
            
            # Get target embeddings (already computed from embedding space)
            target_embeddings = np.array([ex['target_embedding'] for ex in training_examples])
            
            # Train/val split
            val_split = 0.1
            n_val = int(len(training_examples) * val_split)
            n_train = len(training_examples) - n_val
            
            # Shuffle indices
            indices = np.random.permutation(len(training_examples))
            train_indices = indices[:n_train]
            val_indices = indices[n_train:]
            
            train_states = pref_embeddings[train_indices]
            train_targets = target_embeddings[train_indices]
            val_states = pref_embeddings[val_indices]
            val_targets = target_embeddings[val_indices]
            
            print(f"  Training examples: {len(train_indices)}")
            print(f"  Validation examples: {len(val_indices)}")
            
            print(f"\nTraining actor for {PRETRAIN_EPOCHS} epochs...")
            optimizer = torch.optim.Adam(casper_agent.rl_agent.actor.parameters(), lr=0.001)
            batch_size = 32
            
            best_val_loss = float('inf')
            best_epoch = 0
            
            for epoch in tqdm(range(PRETRAIN_EPOCHS), desc="Pretraining epochs", unit="epoch"):
                # Training phase
                casper_agent.rl_agent.actor.train()
                epoch_loss = 0
                num_batches = 0
                
                batch_indices = np.random.permutation(len(train_indices))
                
                for i in range(0, len(batch_indices), batch_size):
                    batch_idx = batch_indices[i:i+batch_size]
                    
                    states_batch = train_states[batch_idx]
                    targets_batch = train_targets[batch_idx]
                    
                    states_tensor = torch.FloatTensor(states_batch)
                    targets_tensor = torch.FloatTensor(targets_batch)
                    
                    predicted = casper_agent.rl_agent.actor(states_tensor)
                    
                    # Normalize embeddings
                    predicted_norm = torch.nn.functional.normalize(predicted, dim=1)
                    targets_norm = torch.nn.functional.normalize(targets_tensor, dim=1)
                    
                    loss = torch.nn.functional.mse_loss(predicted_norm, targets_norm)
                    
                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(casper_agent.rl_agent.actor.parameters(), 1.0)
                    optimizer.step()
                    
                    epoch_loss += loss.item()
                    num_batches += 1
                
                avg_train_loss = epoch_loss / num_batches if num_batches > 0 else 0
                
                # Validation phase
                casper_agent.rl_agent.actor.eval()
                val_loss = 0
                val_cos_sim = 0
                val_batches = 0
                
                with torch.no_grad():
                    for i in range(0, len(val_indices), batch_size):
                        states_batch = val_states[i:i+batch_size]
                        targets_batch = val_targets[i:i+batch_size]
                        
                        states_tensor = torch.FloatTensor(states_batch)
                        targets_tensor = torch.FloatTensor(targets_batch)
                        
                        predicted = casper_agent.rl_agent.actor(states_tensor)
                        
                        predicted_norm = torch.nn.functional.normalize(predicted, dim=1)
                        targets_norm = torch.nn.functional.normalize(targets_tensor, dim=1)
                        
                        batch_loss = torch.nn.functional.mse_loss(predicted_norm, targets_norm)
                        val_loss += batch_loss.item()
                        
                        # Cosine similarity (more interpretable)
                        cos_sim = torch.nn.functional.cosine_similarity(predicted_norm, targets_norm).mean()
                        val_cos_sim += cos_sim.item()
                        
                        val_batches += 1
                
                avg_val_loss = val_loss / val_batches if val_batches > 0 else 0
                avg_val_cos_sim = val_cos_sim / val_batches if val_batches > 0 else 0
                
                # Track best model
                best_marker = ""
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    best_epoch = epoch + 1
                    best_marker = " *"
                    
                    # Save best checkpoint
                    torch.save({
                        'actor': casper_agent.rl_agent.actor.state_dict(),
                        'epoch': epoch + 1,
                        'train_loss': avg_train_loss,
                        'val_loss': avg_val_loss,
                        'val_cos_sim': avg_val_cos_sim,
                        'num_examples': len(train_indices)
                    }, pretrain_checkpoint)
                
                tqdm.write(
                    f"Epoch {epoch+1}/{PRETRAIN_EPOCHS}: "
                    f"train_loss={avg_train_loss:.4f} | "
                    f"val_loss={avg_val_loss:.4f} | "
                    f"val_cos_sim={avg_val_cos_sim:.3f}{best_marker}"
                )
            
            print(f"\nPretraining complete!")
            print(f"  Best epoch: {best_epoch}")
            print(f"  Best val loss: {best_val_loss:.4f}")
            print(f"  Checkpoint saved: {pretrain_checkpoint}")
        
    except Exception as e:
        print(f"\nWARNING: Supervised pretraining failed: {e}")
        print(f"  Actor will train from scratch during RL.")
        import traceback
        traceback.print_exc()

## 7. Define Episode Runners

In [ ]:
from casper.rewards import RecommenderLossReward
import torch
import random

# Calculate baseline loss once (when preferences are empty/unknown)
BASELINE_LOSS_CACHE = {}

def calculate_baseline_loss(agent, user_liked_ids, all_movie_ids, movie_catalog):
    """Calculate baseline BPR loss when user preferences are unknown."""
    cache_key = tuple(sorted(user_liked_ids[:5]))  # Cache by first 5 liked movies
    
    if cache_key in BASELINE_LOSS_CACHE:
        return BASELINE_LOSS_CACHE[cache_key]
    
    # Empty state (no preferences discovered)
    empty_state_text = "likes: unknown | dislikes: unknown"
    empty_state = agent.encoder.encode(empty_state_text, convert_to_numpy=True)
    empty_state_tensor = torch.FloatTensor(empty_state)
    
    # Get positive and negative items
    pos_indices = [movie_catalog.movie_id_to_idx[mid] for mid in user_liked_ids[:10]]
    pos_features = movie_catalog.movie_embeddings_tensor[pos_indices]
    
    num_negatives = min(20, len(all_movie_ids) - len(user_liked_ids))
    neg_movie_ids = random.sample(
        [mid for mid in all_movie_ids if mid not in user_liked_ids],
        num_negatives
    )
    neg_indices = [movie_catalog.movie_id_to_idx[mid] for mid in neg_movie_ids]
    neg_features = movie_catalog.movie_embeddings_tensor[neg_indices]
    
    # Calculate baseline loss
    baseline_loss = agent.recommender.recommender.calculate_bpr_loss(
        empty_state_tensor,
        pos_features,
        neg_features
    )
    
    BASELINE_LOSS_CACHE[cache_key] = baseline_loss
    return baseline_loss


def run_casper_episode(agent, user_profile, evaluator, max_turns=10):
    """
    Run CASPER episode with RecommenderLossReward and enriched logging.

    Reward = -(Loss(t) - Loss(t-1))
    
    For turn 0: reward = -(Loss(0) - baseline_loss) where baseline_loss is measured
    with empty preferences ("likes: unknown | dislikes: unknown")

    Returns:
        (rewards, final_ndcg, turn_logs, user_ground_truth)
    """
    training_profile, test_set = evaluator.prepare_user_profile(user_profile)
    held_out_movie_titles = test_set.get('held_out_movies', [])

    # Convert movie titles to IDs
    title_to_id = {row['title'].split('(')[0].strip(): row['movieId']
                   for _, row in movie_catalog.movies.iterrows()}
    held_out_movies = [title_to_id[title] for title in held_out_movie_titles if title in title_to_id]

    # Get user's liked/disliked movies for ground truth logging
    user_liked_movies = [
        movie for movie, rating in training_profile['ratings'].items()
        if rating >= 4.0
    ]
    user_disliked_movies = [
        movie for movie, rating in training_profile['ratings'].items()
        if rating <= 2.0
    ]
    liked_movie_ids = [title_to_id[title] for title in user_liked_movies if title in title_to_id]
    
    # User ground truth for episode-level logging
    user_ground_truth = {
        'user_id': user_profile.get('user_id', 'unknown'),
        'num_ratings': len(training_profile['ratings']),
        'liked_movies': user_liked_movies[:10],  # Top 10 for readability
        'disliked_movies': user_disliked_movies[:5],  # Top 5 dislikes
        'held_out_movies': held_out_movie_titles
    }
    
    all_movie_ids = movie_catalog.movies['movieId'].tolist()
    num_negatives = min(20, len(all_movie_ids) - len(liked_movie_ids))

    # Calculate baseline loss for first turn
    baseline_loss = calculate_baseline_loss(agent, liked_movie_ids, all_movie_ids, movie_catalog)

    agent.reset_conversation()
    reward_fn = RecommenderLossReward()
    
    rewards = []
    turn_logs = []
    prev_loss = baseline_loss  # Start from baseline

    for turn in range(max_turns):
        # Ask question with debug info for logging
        question, rl_debug = agent.ask_question(explore=True, verbose=False, return_debug_info=True)

        # User response with tool call logging
        response, tool_calls = user_sim.simulate_response(
            question=question,
            user_profile=training_profile,
            conversation_history=agent.conversation_history,
            return_tool_calls=True
        )

        agent.process_user_response(response, user_target_movies=None)

        # Calculate BPR loss
        current_loss = 0.0
        if len(liked_movie_ids) > 0:
            try:
                state = agent.encode_conversation_state()
                state_tensor = torch.FloatTensor(state)

                pos_indices = [movie_catalog.movie_id_to_idx[mid] for mid in liked_movie_ids[:10]]
                pos_features = movie_catalog.movie_embeddings_tensor[pos_indices]

                neg_movie_ids = random.sample(
                    [mid for mid in all_movie_ids if mid not in liked_movie_ids],
                    num_negatives
                )
                neg_indices = [movie_catalog.movie_id_to_idx[mid] for mid in neg_movie_ids]
                neg_features = movie_catalog.movie_embeddings_tensor[neg_indices]

                current_loss = agent.recommender.recommender.calculate_bpr_loss(
                    state_tensor,
                    pos_features,
                    neg_features
                )
            except Exception as e:
                print(f"Warning: Could not calculate loss: {e}")
                current_loss = prev_loss  # Keep previous loss on error

        # Calculate reward as delta from previous loss
        reward = -(current_loss - prev_loss)
        rewards.append(reward)
        prev_loss = current_loss  # Update for next turn

        # NDCG for logging (every 3 turns)
        current_ndcg = 0.0
        if turn % 3 == 0 and len(held_out_movies) > 0:
            try:
                recommendations = agent.recommend_movies(top_k=10)
                recommended_ids = [movie_id for movie_id, _, _ in recommendations]
                current_ndcg = agent._calculate_ndcg_at_k(recommended_ids, held_out_movies, k=10)
                agent.ndcg_history.append(current_ndcg)
            except:
                pass

        # Get recommendations for logging
        try:
            recommendations = agent.recommend_movies(top_k=10)
            rec_titles = [title for _, title, _ in recommendations[:5]]
        except:
            rec_titles = []

        # Enriched turn log with RL details and tool calls
        turn_log = {
            "turn": turn + 1,
            "agent_question": question,
            "rl_prediction": rl_debug,  # Embedding sample + nearest entities
            "user_response": response,
            "user_tool_calls": tool_calls,  # Function calls made by user simulator
            "preferences_extracted": dict(agent.discovered_preferences),
            "top_recommendations": rec_titles,
            "ndcg": current_ndcg if current_ndcg > 0 else (agent.ndcg_history[-1] if agent.ndcg_history else 0.0),
            "loss": current_loss,
            "reward": reward
        }
        turn_logs.append(turn_log)

    final_ndcg = agent.ndcg_history[-1] if agent.ndcg_history else 0.0
    return rewards, final_ndcg, turn_logs, user_ground_truth


def run_baseline_episode(agent, user_profile, evaluator, shared_recommender, shared_pref_extractor, max_turns=10):
    """Run baseline episode - compute recommendations only at end."""
    training_profile, test_set = evaluator.prepare_user_profile(user_profile)
    held_out_movie_titles = test_set.get('held_out_movies', [])
    
    title_to_id = {row['title'].split('(')[0].strip(): row['movieId'] 
                   for _, row in movie_catalog.movies.iterrows()}
    held_out_movies = [title_to_id[title] for title in held_out_movie_titles if title in title_to_id]
    
    agent.reset_conversation()
    
    for turn in range(max_turns):
        question = agent.ask_question(verbose=False)
        response = user_sim.simulate_response(
            question=question,
            user_profile=training_profile,
            conversation_history=agent.conversation_history
        )
        agent.process_user_response(response)
    
    discovered_prefs = agent.discovered_preferences
    pref_text = shared_pref_extractor.preferences_to_text(discovered_prefs)
    
    if pref_text:
        state = agent.encoder.encode(pref_text, convert_to_numpy=True)
    else:
        state = np.zeros(384)
    
    recommendations = shared_recommender.recommend(state, top_k=10)
    recommended_ids = [movie_id for movie_id, _, _ in recommendations]
    final_ndcg = metrics.ndcg_at_k(recommended_ids, held_out_movies, k=10)
    
    return [], final_ndcg, []

print("Episode runners defined")
print("Using RecommenderLossReward with baseline fix:")
print("  Turn 0: reward = -(Loss(0) - baseline_loss)")
print("  Turn 1+: reward = -(Loss(t) - Loss(t-1))")
print("  Baseline loss = loss when preferences = 'unknown'")
print("\nEnriched logging enabled:")
print("  - RL prediction details (embedding + nearest entities)")
print("  - User simulator tool calls (query_rating, search_ratings, etc.)")
print("  - User ground truth (liked/disliked/held-out movies)")

In [ ]:
from tqdm import tqdm
import json

episode_rewards = []
episode_ndcg = []
train_losses = []
start_episode = 0

# Create conversation log files
conv_log_path = CHECKPOINT_DIR / 'training_conversations.jsonl'
detailed_log_path = CHECKPOINT_DIR / 'training_detailed.jsonl'

conv_log_file = open(conv_log_path, 'a', encoding='utf-8')
detailed_log_file = open(detailed_log_path, 'a', encoding='utf-8')

rl_checkpoints = sorted(CHECKPOINT_DIR.glob('rl_episode_*.pt'))
if rl_checkpoints:
    latest_checkpoint = rl_checkpoints[-1]
    print(f"Loading RL checkpoint from {latest_checkpoint}")
    checkpoint = torch.load(latest_checkpoint)
    casper_agent.rl_agent.actor.load_state_dict(checkpoint['actor'])
    casper_agent.rl_agent.critic.load_state_dict(checkpoint['critic'])
    start_episode = checkpoint['episode']
    episode_rewards = checkpoint['rewards']
    episode_ndcg = checkpoint['ndcg']
    print(f"Resuming from episode {start_episode}")

print(f"Starting RL training from episode {start_episode}...")
print(f"Conversation log: {conv_log_path}")
print(f"Detailed log (turn-by-turn): {detailed_log_path}")

# Progress bar with live metrics
pbar = tqdm(range(start_episode, RL_EPISODES), desc="RL Training", unit="ep", initial=start_episode, total=RL_EPISODES)

try:
    for ep in pbar:
        profile = user_sim.sample_user()
        
        # Run episode with enriched logging (now returns user_ground_truth)
        rewards, final_ndcg, turn_logs, user_ground_truth = run_casper_episode(
            agent=casper_agent,
            user_profile=profile,
            evaluator=evaluator,
            max_turns=MAX_TURNS
        )
        
        train_metrics = casper_agent.train_from_episode(use_per_turn_rewards=True)
        
        episode_rewards.append(sum(rewards) if rewards else 0)
        episode_ndcg.append(final_ndcg)
        actor_loss = 0
        if train_metrics:
            actor_loss = train_metrics.get('actor_loss', 0)
            train_losses.append(actor_loss)
        
        # Log summary conversation to simple log (with user ground truth)
        conv_log = {
            'episode': ep + 1,
            'user_ground_truth': user_ground_truth,  # NEW: User's actual preferences
            'conversation': casper_agent.conversation_history,
            'discovered_preferences': casper_agent.discovered_preferences,
            'reward': sum(rewards) if rewards else 0,
            'ndcg': final_ndcg,
            'actor_loss': actor_loss
        }
        conv_log_file.write(json.dumps(conv_log) + '\n')
        conv_log_file.flush()
        
        # Log detailed turn-by-turn info to detailed log (includes RL + tool calls)
        detailed_log = {
            'episode': ep + 1,
            'user_ground_truth': user_ground_truth,  # NEW: User's actual preferences
            'turns': turn_logs,  # Already includes rl_prediction and user_tool_calls
            'final_metrics': {
                'total_reward': sum(rewards) if rewards else 0,
                'final_ndcg': final_ndcg,
                'actor_loss': actor_loss
            }
        }
        detailed_log_file.write(json.dumps(detailed_log) + '\n')
        detailed_log_file.flush()
        
        # Update progress bar
        pbar.set_postfix({
            'reward': f'{episode_rewards[-1]:.3f}',
            'NDCG': f'{final_ndcg:.3f}',
            'loss': f'{actor_loss:.3f}'
        })
        
        # Save CURRENT checkpoint every episode (overwrites)
        current_checkpoint = CHECKPOINT_DIR / 'rl_current.pt'
        torch.save({
            'actor': casper_agent.rl_agent.actor.state_dict(),
            'critic': casper_agent.rl_agent.critic.state_dict(),
            'episode': ep + 1,
            'rewards': episode_rewards,
            'ndcg': episode_ndcg
        }, current_checkpoint)
        
        # Save INTERMEDIATE checkpoint every 100 episodes (keeps history)
        if (ep + 1) % 100 == 0:
            checkpoint_path = CHECKPOINT_DIR / f'rl_episode_{ep+1}.pt'
            torch.save({
                'actor': casper_agent.rl_agent.actor.state_dict(),
                'critic': casper_agent.rl_agent.critic.state_dict(),
                'episode': ep + 1,
                'rewards': episode_rewards,
                'ndcg': episode_ndcg
            }, checkpoint_path)
            tqdm.write(f"Milestone checkpoint saved: {checkpoint_path}")

except KeyboardInterrupt:
    print("\n\nTraining interrupted by user!")
    print("Saving current state...")
    
    # Save interrupt checkpoint
    interrupt_checkpoint = CHECKPOINT_DIR / f'rl_interrupted_ep{ep+1}.pt'
    torch.save({
        'actor': casper_agent.rl_agent.actor.state_dict(),
        'critic': casper_agent.rl_agent.critic.state_dict(),
        'episode': ep + 1,
        'rewards': episode_rewards,
        'ndcg': episode_ndcg,
        'interrupted': True
    }, interrupt_checkpoint)
    print(f"Interrupt checkpoint saved: {interrupt_checkpoint}")
    print(f"Episodes completed: {ep + 1}/{RL_EPISODES}")
    
finally:
    conv_log_file.close()
    detailed_log_file.close()
    pbar.close()

print(f"\nRL training complete")
print(f"  Episodes completed: {len(episode_rewards)}")
if len(episode_rewards) >= 100:
    print(f"  Final reward (last 100): {np.mean(episode_rewards[-100:]):.3f}")
    print(f"  Final NDCG (last 100): {np.mean(episode_ndcg[-100:]):.3f}")
else:
    print(f"  Final reward (all): {np.mean(episode_rewards):.3f}")
    print(f"  Final NDCG (all): {np.mean(episode_ndcg):.3f}")

## 9. Load Best Checkpoint for Evaluation

In [ ]:
# Load most recent checkpoint for evaluation
# Priority: rl_current.pt > rl_interrupted_*.pt > rl_episode_*.pt

checkpoints_to_try = [
    CHECKPOINT_DIR / 'rl_current.pt',  # Most recent (saved every episode)
    *sorted(CHECKPOINT_DIR.glob('rl_interrupted_*.pt'), reverse=True),  # Interrupted checkpoints
    *sorted(CHECKPOINT_DIR.glob('rl_episode_*.pt'), reverse=True)  # Milestone checkpoints
]

loaded = False
for checkpoint_path in checkpoints_to_try:
    if checkpoint_path.exists():
        print(f"Loading checkpoint for evaluation: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        casper_agent.rl_agent.actor.load_state_dict(checkpoint['actor'])
        casper_agent.rl_agent.critic.load_state_dict(checkpoint['critic'])
        print(f"Model trained for {checkpoint['episode']} episodes")
        
        if 'ndcg' in checkpoint and len(checkpoint['ndcg']) > 0:
            recent_ndcg = np.mean(checkpoint['ndcg'][-min(10, len(checkpoint['ndcg'])):])
            print(f"Training NDCG (last 10 eps): {recent_ndcg:.3f}")
        
        loaded = True
        break

if not loaded:
    print("WARNING: No checkpoints found - using untrained agent!")
    print("This will produce poor results.")

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'Agent': ['CASPER (ours)', 'PureLLMAgent', 'RandomAgent'],
    'NDCG@10': [
        f"{casper_results['ndcg_mean']:.3f} ± {casper_results['ndcg_std']:.3f}",
        f"{llm_results['ndcg_mean']:.3f} ± {llm_results['ndcg_std']:.3f}",
        f"{random_results['ndcg_mean']:.3f} ± {random_results['ndcg_std']:.3f}"
    ],
    'Total Reward': [
        f"{casper_results['reward_mean']:.3f} ± {casper_results['reward_std']:.3f}",
        f"{llm_results['reward_mean']:.3f} ± {llm_results['reward_std']:.3f}",
        f"{random_results['reward_mean']:.3f} ± {random_results['reward_std']:.3f}"
    ]
})

print("\n" + "="*60)
print("RESULTS")
print("="*60 + "\n")
print(results_df.to_string(index=False))
print("\n" + "="*60)

# Save results
results_path = CHECKPOINT_DIR / 'results.csv'
results_df.to_csv(results_path, index=False)
print(f"\nResults saved: {results_path}")

## 10. Results Comparison

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('RL Training: Rewards', 'RL Training: Recommendation Quality')
)

# Rewards plot
fig.add_trace(
    go.Scatter(y=episode_rewards, mode='lines', name='Episode', opacity=0.3, line=dict(color='blue')),
    row=1, col=1
)

# Moving average
window = 50
if len(episode_rewards) >= window:
    smooth_rewards = pd.Series(episode_rewards).rolling(window).mean()
    fig.add_trace(
        go.Scatter(y=smooth_rewards, mode='lines', name=f'{window}-episode MA', line=dict(color='blue', width=2)),
        row=1, col=1
    )

# NDCG plot
fig.add_trace(
    go.Scatter(y=episode_ndcg, mode='lines', name='Episode', opacity=0.3, line=dict(color='red'), showlegend=False),
    row=1, col=2
)

if len(episode_ndcg) >= window:
    smooth_ndcg = pd.Series(episode_ndcg).rolling(window).mean()
    fig.add_trace(
        go.Scatter(y=smooth_ndcg, mode='lines', name=f'{window}-episode MA', line=dict(color='red', width=2), showlegend=False),
        row=1, col=2
    )

fig.update_xaxes(title_text="Episode", row=1, col=1)
fig.update_xaxes(title_text="Episode", row=1, col=2)
fig.update_yaxes(title_text="Total Reward", row=1, col=1)
fig.update_yaxes(title_text="NDCG@10", row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

# Save plot
plot_path = CHECKPOINT_DIR / 'training_progress.html'
fig.write_html(str(plot_path))
print(f"Plot saved: {plot_path}")

## 11. Training Visualization

In [ ]:
final_checkpoint = CHECKPOINT_DIR / 'casper_final.pt'

checkpoint_data = {
    'actor': casper_agent.rl_agent.actor.state_dict(),
    'critic': casper_agent.rl_agent.critic.state_dict(),
    'num_episodes': RL_EPISODES,
    'episode_rewards': episode_rewards,
    'episode_ndcg': episode_ndcg,
    'config': config
}

torch.save(checkpoint_data, final_checkpoint)
print(f"Final checkpoint saved: {final_checkpoint}")
print(f"\nTraining complete!")
print(f"  Total episodes: {RL_EPISODES}")
print(f"  Final NDCG@10: {np.mean(episode_ndcg[-100:]):.3f}")
print(f"  Improvement over random: {casper_results['ndcg_mean'] - random_results['ndcg_mean']:.3f}")